In [ ]:
import warnings

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

from itertools import combinations

import numpy as np
from astropy import units as u
from astropy.coordinates import ICRS
from astropy.table import QTable
from astropy_healpix import HEALPix
from ligo.skymap import plot  # noqa: F401
from ligo.skymap.util import progress_map
from m4opt.fov import footprint_healpix
from m4opt.missions import uvex as mission
from m4opt.utils.numpy import count_intersect1d
from matplotlib import pyplot as plt
from regions import Regions
from tqdm.auto import tqdm

In [ ]:
(bounding_rectangle,) = Regions.read("../fov/bounding-rectangle.ds9")
(inscribed_circle,) = Regions.read("../fov/inscribed-circle.ds9")
plan = QTable.read("../tables/initial-survey.ecsv")

## Time usage

In [ ]:
total_time = u.Quantity(list(plan.meta["total_time"].values())).to(u.day)
action = list(plan.meta["total_time"].keys())
plt.pie(
    total_time,
    labels=action,
    autopct=lambda pct: (0.01 * pct * total_time.sum().to(u.day)).round(2),
)
plt.savefig("../visualizations/time-utilization.pdf", metadata={"CreationDate": None})

## Fraction of sky visited N times

In [ ]:
obs = plan[plan["action"] == "observe"]
hpx = HEALPix(nside=2048, frame=ICRS())

fovs = [bounding_rectangle, inscribed_circle, mission.fov]
fov_names = ["Bounding rectangle", "Inscribed circle", "Chips"]
visits = []

for fov in tqdm(fovs):
    footprints = footprint_healpix(hpx, fov, obs["target_coord"], obs["roll"])
    visits.append(
        np.bincount(np.bincount(np.concatenate(footprints), minlength=hpx.npix))
    )
max_visits = max(len(v) for v in visits)
visits = np.stack([np.pad(v, (0, max_visits - len(v))) for v in visits])

In [ ]:
table = QTable(
    {
        "FOV": fov_names,
        **{str(i): col for i, col in enumerate(visits.T / hpx.npix * 100 * u.percent)},
    }
)
for colname in table.colnames:
    if colname != "FOV":
        table[colname].info.format = "%.2f"
table

## Area of sky visited N times

In [ ]:
table = QTable(
    {
        "FOV": fov_names,
        **{
            str(i): col
            for i, col in enumerate((visits.T * hpx.pixel_area).to(u.deg**2))
        },
    }
)
for colname in table.colnames:
    if colname != "FOV":
        table[colname].info.format = "%.0f"
table

## Cadence distribution

In [ ]:
start_mjd = obs["start_time"].mjd
weights = np.asarray(
    list(progress_map(count_intersect1d, *zip(*combinations(footprints, 2)), jobs=None))
)
delays = np.asarray([abs(a - b) for a, b in combinations(start_mjd, 2)])

In [ ]:
v = np.bincount(np.bincount(np.concatenate(footprints), minlength=hpx.npix))
area_visited_at_most_once = (v[0] + v[1]) * hpx.pixel_area.to_value(u.deg**2)

for nbins in [2, 4, 8, 20, 40]:
    fig, axs = plt.subplots(1, 2, sharey=True, width_ratios=(20, 1))
    axs[0].hist(
        delays,
        weights=weights * hpx.pixel_area.to_value(u.deg**2),
        bins=np.logspace(
            np.floor(np.log10(delays.min())), np.ceil(np.log10(delays.max())), nbins + 1
        ),
    )
    axs[0].set_xscale("log")
    axs[0].set_xlabel("Time delay (days)")
    axs[0].set_ylabel("Area (deg$^2$)")
    axs[0].set_title("Cadence distribution")
    axs[0].grid()

    axs[1].bar("Visited\nat most once", area_visited_at_most_once)

    axs[0].spines.left.set_visible(False)
    axs[0].spines.top.set_visible(False)
    axs[0].spines.right.set_visible(False)
    axs[1].spines.left.set_visible(False)
    axs[1].spines.top.set_visible(False)
    axs[1].spines.right.set_visible(False)
    fig.savefig(
        f"../visualizations/cadence-distribution-{nbins}-bins.pdf",
        metadata={"CreationDate": None},
    )